# 协同过滤：从记忆到推理

协同过滤（Collaborative Filtering, CF）是推荐系统中最经典、应用最广泛的算法家族。其核心思想源于社会心理学中的“从众效应”：**相似的人倾向于喜欢相似的东西**。

与基于内容的推荐不同，CF 完全忽略物品的内容属性（如标题、标签），仅依赖**用户-物品交互矩阵（User-Item Interaction Matrix）**。这种“无模型”（Model-Free）的特性使其能够挖掘出很多令人意想不到的关联（Serendipity），但也面临着**数据稀疏**和**冷启动**的严峻挑战。

本 Notebook 将深入剖析 User-Based 和 Item-Based 两种主流范式，并提供代码实现。


## 1. 核心范式：UserCF vs ItemCF

协同过滤主要分为基于用户的协同过滤（User-Based CF）和基于物品的协同过滤（Item-Based CF）。这两种方法分别从用户相似度和物品相似度的角度出发，利用群体智慧来发现潜在的兴趣偏好。虽然出发点不同，但它们在数学形式上具有高度的对称性。

### 1.1 User-Based CF (基于用户)

**核心逻辑**：寻找和你口味相似的“邻居”，把他们喜欢但你没看过的东西推荐给你。

- **适用场景**：社交属性强、新闻资讯等用户兴趣变化快且物品更新极快的场景。
- **工程挑战**：用户数（M）通常远大于物品数（N），维护 $M \times M$ 的用户相似度矩阵代价高昂。

### 1.2 Item-Based CF (基于物品)

**核心逻辑**：寻找和你过去喜欢的物品相似的物品。这里的“相似”是指“经常被同一拨人一起喜欢”。

- **适用场景**：电商、电影、音乐等物品属性稳定、Item 数相对固定的场景（如 Amazon, Netflix）。
- **优势**：物品相似度矩阵 $N \times N$ 相对稳定，可离线计算并缓存。**工业界绝大多数场景首选 ItemCF。**


## 2. 相似度度量

相似度计算是 CF 的灵魂。给定两个向量 $\vec{a}$ 和 $\vec{b}$，常用的度量标准如下：

### 2.1 余弦相似度

衡量向量在方向上的差异，不敏感于绝对数值大小。

$$
\text{sim}(\vec{a}, \vec{b}) = \cos(\theta) = \frac{\vec{a} \cdot \vec{b}}{||\vec{a}|| \cdot ||\vec{b}||}
$$

### 2.2 皮尔逊相关系数

去中心化的余弦相似度。通过减去用户均值，消除用户打分尺度的差异（有的用户宽容全打 5 分，有的苛刻只打 3 分）。

$$
\text{sim}(u, v) = \frac{\sum_{i} (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_{i} (r_{ui} - \bar{r}_u)^2} \sqrt{\sum_{i} (r_{vi} - \bar{r}_v)^2}}
$$

### 2.3 杰卡德相似度

适用于隐式反馈（0/1 数据，如点击/未点击）。

$$
\text{sim}(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$


## 3. 示例计算：手工算一次协同过滤

为了更直观地理解协同过滤的计算过程，我们以 **User-Based CF** 为例，通过代码模拟一次预测过程。

### 3.1 场景假设

假设有 4 个用户和 5 部电影，评分数据如下（NaN 表示未评分）：

| 用户 | 电影 A | 电影 B | 电影 C | 电影 D | 电影 E |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **User 1 (Target)** | **5** | **3** | **4** | **?** | **-** |
| User 2 | 3 | 1 | 2 | 3 | 3 |
| User 3 | 4 | 3 | 4 | 3 | 5 |
| User 4 | 3 | 3 | 1 | 5 | 4 |

**目标**：预测 **User 1** 对 **电影 D** 的评分。

### 3.2 代码验证


In [1]:
import numpy as np
import pandas as pd

# 1. 准备数据
users = ['User 1', 'User 2', 'User 3', 'User 4']
movies = ['A', 'B', 'C', 'D', 'E']
data_manual = np.array([
    [5, 3, 4, np.nan, np.nan], # User 1
    [3, 1, 2, 3, 3],           # User 2
    [4, 3, 4, 3, 5],           # User 3
    [3, 3, 1, 5, 4]            # User 4
])

df_manual = pd.DataFrame(data_manual, index=users, columns=movies)
print("原始评分矩阵：")
print(df_manual)

原始评分矩阵：
          A    B    C    D    E
User 1  5.0  3.0  4.0  NaN  NaN
User 2  3.0  1.0  2.0  3.0  3.0
User 3  4.0  3.0  4.0  3.0  5.0
User 4  3.0  3.0  1.0  5.0  4.0


In [2]:
# 2. 计算用户相似度 (Pearson Correlation)
# 目标：计算 User 1 与其他用户的相似度
# 注意：只基于共同评分的物品 (A, B, C) 进行计算

target_user = 'User 1'
other_users = ['User 2', 'User 3', 'User 4']
common_items = ['A', 'B', 'C']

target_ratings = df_manual.loc[target_user, common_items].values
# User 1 均值 (基于 A,B,C)
mean_target = np.mean(target_ratings)
print(f"\nUser 1 在共同项目上的均值: {mean_target:.2f}")

similarities = {}

print("\n相似度计算详情：")
for user in other_users:
    other_ratings = df_manual.loc[user, common_items].values
    mean_other = np.mean(other_ratings)
    
    # 去均值
    vec_target = target_ratings - mean_target
    vec_other = other_ratings - mean_other
    
    # 计算余弦相似度 (即去均值后的 Pearson)
    numerator = np.dot(vec_target, vec_other)
    denominator = np.linalg.norm(vec_target) * np.linalg.norm(vec_other)
    
    if denominator == 0:
        sim = 0
    else:
        sim = numerator / denominator
        
    similarities[user] = sim
    print(f"Sim({target_user}, {user}): {sim:.2f}")


User 1 在共同项目上的均值: 4.00

相似度计算详情：
Sim(User 1, User 2): 1.00
Sim(User 1, User 3): 0.87
Sim(User 1, User 4): 0.00


In [4]:
# 3. 生成推荐 (加权平均)
# 选取相似度最高的 2 个邻居 (User 2, User 3) 来预测 Item D

top_neighbors = ['User 2', 'User 3']
target_item = 'D'

print("\n预测计算详情：")
weighted_sum = 0
sim_sum = 0

for neighbor in top_neighbors:
    sim = similarities[neighbor]
    rating = df_manual.loc[neighbor, target_item]
    
    # 注意：这里的均值通常使用用户的所有评分计算，以反映其整体打分习惯
    neighbor_all_ratings = df_manual.loc[neighbor].dropna().values
    mean_neighbor = np.mean(neighbor_all_ratings)
    
    # 评分偏差 (Bias)
    bias = rating - mean_neighbor
    
    weighted_sum += sim * bias
    sim_sum += abs(sim)
    
    print(f"{neighbor}: Sim={sim:.2f}, Rating={rating}, Mean={mean_neighbor:.1f}, Bias={bias:.1f}")

# 最终预测公式: Mean_Target + (Sum(Sim * Bias) / Sum(|Sim|))
pred_score = mean_target + weighted_sum / sim_sum

print(f"\n预测 User 1 对 Item D 的评分: {pred_score:.2f}")


预测计算详情：
User 2: Sim=1.00, Rating=3.0, Mean=2.4, Bias=0.6
User 3: Sim=0.87, Rating=3.0, Mean=3.8, Bias=-0.8

预测 User 1 对 Item D 的评分: 3.95


## 4. 工程实践：Item-Based CF 实现

我们将实现一个基于 Item-Based CF 的评分预测系统。

### 4.1 准备数据


In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 模拟用户-物品评分矩阵 (User-Item Matrix)
# 行：用户，列：物品 (A, B, C, D, E)
# 0 表示未评分
data = np.array([
    [5, 3, 4, 0, 2],  # User 1
    [3, 1, 2, 3, 3],  # User 2
    [4, 3, 4, 3, 5],  # User 3
    [3, 3, 1, 5, 4],  # User 4
    [1, 5, 5, 2, 1]   # User 5
])

items = ['A', 'B', 'C', 'D', 'E']
df = pd.DataFrame(data, columns=items)
print("原始评分矩阵:\n", df)


原始评分矩阵:
    A  B  C  D  E
0  5  3  4  0  2
1  3  1  2  3  3
2  4  3  4  3  5
3  3  3  1  5  4
4  1  5  5  2  1


### 4.2 物品相似度计算

注意：Item-Based CF 需要计算物品向量（列向量）之间的相似度，因此需要对矩阵进行转置或直接计算列相似度。


In [6]:
# 计算物品相似度 (Item-Item Similarity)
# 实际生产中，对于大规模稀疏矩阵，通常只保留 Top-K 相似度，并使用稀疏存储
item_sim_matrix = cosine_similarity(df.T)
item_sim_df = pd.DataFrame(item_sim_matrix, index=items, columns=items)

print("\n物品相似度矩阵:\n", item_sim_df.round(2))


物品相似度矩阵:
       A     B     C     D     E
A  1.00  0.78  0.82  0.72  0.91
B  0.78  1.00  0.94  0.74  0.76
C  0.82  0.94  1.00  0.61  0.74
D  0.72  0.74  0.61  1.00  0.90
E  0.91  0.76  0.74  0.90  1.00


### 4.3 评分预测

预测公式：

$$
\hat{r}_{ui} = \frac{\sum_{j \in S(i, K)} w_{ij} \cdot r_{uj}}{\sum_{j \in S(i, K)} |w_{ij}|}
$$

其中 $S(i, K)$ 是与物品 $i$ 最相似的 $K$ 个物品集合，$w_{ij}$ 是相似度。


In [8]:
def predict_rating(user_id, item_id, data, sim_matrix, k=2):
    user_ratings = data[user_id, :]
    
    # 找到用户已评分的物品索引
    rated_indices = np.where(user_ratings > 0)[0]
    
    # 如果用户没评过任何分，无法进行 CF 推荐（冷启动）
    if len(rated_indices) == 0:
        return 0
        
    # 获取该物品与其他物品的相似度
    sim_scores = sim_matrix[item_id, rated_indices]
    
    # 找到相似度最高的 K 个物品 (Top-K Neighbors)
    # argsort 返回从小到大的索引，取最后 K 个并反转
    if len(sim_scores) > k:
        top_k_idx = np.argsort(sim_scores)[-k:]
        top_k_sim = sim_scores[top_k_idx]
        top_k_ratings = user_ratings[rated_indices][top_k_idx]
    else:
        top_k_sim = sim_scores
        top_k_ratings = user_ratings[rated_indices]
        
    # 加权平均预测
    if np.sum(top_k_sim) == 0:
        return 0
        
    pred_score = np.dot(top_k_sim, top_k_ratings) / np.sum(top_k_sim)
    return pred_score

# 预测 User 1 对物品 D (索引 3) 的评分
user_idx = 0
item_idx = 3 # Item D
pred = predict_rating(user_idx, item_idx, data, item_sim_matrix, k=2)

print(f"\n预测 User 1 对 Item D 的评分: {pred:.2f}")
print(f"User 1 的真实历史评分: {data[user_idx]}")


预测 User 1 对 Item D 的评分: 2.45
User 1 的真实历史评分: [5 3 4 0 2]


## 5. 关键问题与优化

尽管协同过滤算法思想简单且有效，但在实际的大规模工业应用中，我们面临着数据稀疏、热门物品干扰以及计算扩展性等诸多挑战。

### 5.1 数据稀疏性
- **问题**：两个用户可能没有任何共同评分，导致相似度为 0。
- **对策**：矩阵填充、降维（矩阵分解）。

### 5.2 热门物品惩罚
- **问题**：热门物品（如《哈利波特》）会被绝大多数人购买。如果两个用户都买了热门物品，并不能说明他们兴趣相似。
- **改进**：在计算相似度时，对热门物品进行降权惩罚（IUF, Inverse User Frequency）。

### 5.3 计算性能优化
- **问题**：计算所有物品两两相似度的时间复杂度是 $O(N^2)$。
- **对策**：使用**倒排索引 (Inverted Index)**，只计算有过共同交互的物品对。


## 6. 总结

| 特性 | User-Based CF | Item-Based CF |
| :--- | :--- | :--- |
| **推荐逻辑** | 兴趣相投的朋友推荐 | 买了又买 |
| **实时性** | 新用户加入需重算相似度 | 新物品加入需重算相似度 |
| **解释性** | 弱 ("和你相似的人都看了") | 强 ("因为你看了 A，所以推荐 B") |
| **冷启动** | 对新用户不友好 | 对新物品不友好 |
| **稳定性** | 差 (用户兴趣多变) | 好 (物品关系相对固定) |
| **工业界应用** | 社交、新闻 | 电商、视频、音乐 (主流) |

在下一章中，我们将介绍 **矩阵分解 (Matrix Factorization)**，它从数学上优雅地解决了稀疏性问题，并将协同过滤推向了隐语义模型的新高度。
